Code generating mockup tube positions

In [1]:
import numpy as np
n=''
x=''
y=''
z=''
angle=np.pi/50
for hi in range(49):
    n+=f'{hi}, '
    x+=f'{np.cos(angle*hi)}, '
    y+=f'{np.sin(angle*hi)}, '
    z+=f'0, '
print(n[:-2])
print(x[:-2])
print(y[:-2])
print(z[:-2])

0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48
1.0, 0.9980267284282716, 0.9921147013144779, 0.9822872507286887, 0.9685831611286311, 0.9510565162951535, 0.9297764858882513, 0.9048270524660195, 0.8763066800438636, 0.8443279255020151, 0.8090169943749475, 0.7705132427757891, 0.7289686274214116, 0.6845471059286886, 0.6374239897486896, 0.587785252292473, 0.5358267949789965, 0.48175367410171516, 0.42577929156507266, 0.36812455268467786, 0.30901699437494745, 0.24868988716485474, 0.18738131458572452, 0.12533323356430426, 0.0627905195293133, -1.6081226496766366e-16, -0.0627905195293134, -0.12533323356430437, -0.18738131458572482, -0.24868988716485485, -0.30901699437494756, -0.368124552684678, -0.4257792915650727, -0.48175367410171543, -0.5358267949789969, -0.587785252292473, -0.6374239897486897, -0.6845471059286887, -0.7289686274214117, -0.7705132427757894, -0

Function to calculate the center of the analyzer panels based on the sample-corner distances

In [2]:
import numpy as np


def analyzer_geometry_with_tiles(
    d_ll,
    d_lr,
    d_ur,
    horizontal_offset_deg,
    vertical_offset_deg=0.0,
    horizontal_coverage_deg=5.0,
    vertical_coverage_deg=10.0,
    crystal_size=0.10,
):
    """
    Generate spherical analyzer geometry and hexagonal crystal positions.

    Returns crystal positions relative to analyzer center as separate lists.
    """

    # ------------------------------------------------------------
    # Angular setup
    # ------------------------------------------------------------

    theta_h = np.radians(horizontal_offset_deg)
    theta_v = np.radians(vertical_offset_deg)

    h_half = np.radians(horizontal_coverage_deg / 2.0)
    v_half = np.radians(vertical_coverage_deg / 2.0)

    # ------------------------------------------------------------
    # Central analyzer direction
    # ------------------------------------------------------------

    center_dir = np.array([
        np.cos(theta_v) * np.cos(theta_h),
        np.cos(theta_v) * np.sin(theta_h),
        np.sin(theta_v)
    ])

    center_dir /= np.linalg.norm(center_dir)

    # ------------------------------------------------------------
    # Local coordinate system
    # ------------------------------------------------------------

    horizontal_axis = np.array([
        -np.sin(theta_h),
        np.cos(theta_h),
        0.0
    ])

    horizontal_axis /= np.linalg.norm(horizontal_axis)

    vertical_axis = np.cross(center_dir, horizontal_axis)
    vertical_axis /= np.linalg.norm(vertical_axis)

    # ------------------------------------------------------------
    # Corner directions
    # ------------------------------------------------------------

    dir_ll = (
        center_dir
        - np.tan(h_half) * horizontal_axis
        - np.tan(v_half) * vertical_axis
    )
    dir_ll /= np.linalg.norm(dir_ll)

    dir_lr = (
        center_dir
        + np.tan(h_half) * horizontal_axis
        - np.tan(v_half) * vertical_axis
    )
    dir_lr /= np.linalg.norm(dir_lr)

    dir_ur = (
        center_dir
        + np.tan(h_half) * horizontal_axis
        + np.tan(v_half) * vertical_axis
    )
    dir_ur /= np.linalg.norm(dir_ur)

    # ------------------------------------------------------------
    # Corner positions
    # ------------------------------------------------------------

    p_ll = d_ll * dir_ll
    p_lr = d_lr * dir_lr
    p_ur = d_ur * dir_ur

    # ------------------------------------------------------------
    # Analyzer center
    # ------------------------------------------------------------

    center = (p_ll + p_lr + p_ur) / 3.0

    # ------------------------------------------------------------
    # Surface normal
    # ------------------------------------------------------------

    v1 = p_lr - p_ll
    v2 = p_ur - p_lr

    normal = np.cross(v1, v2)
    normal /= np.linalg.norm(normal)

    if np.dot(normal, center) > 0:
        normal *= -1

    # ------------------------------------------------------------
    # Orientation angles
    # ------------------------------------------------------------

    yaw_deg = np.degrees(np.arctan2(normal[1], normal[0]))

    pitch_deg = np.degrees(
        np.arctan2(
            normal[2],
            np.sqrt(normal[0] ** 2 + normal[1] ** 2)
        )
    )

    # ------------------------------------------------------------
    # Physical analyzer dimensions
    # ------------------------------------------------------------

    avg_distance = np.linalg.norm(center)

    width = (
        2.0 * avg_distance *
        np.tan(np.radians(horizontal_coverage_deg / 2.0))
    )

    height = (
        2.0 * avg_distance *
        np.tan(np.radians(vertical_coverage_deg / 2.0))
    )

    # ------------------------------------------------------------
    # Hexagon packing
    # ------------------------------------------------------------

    hex_width = crystal_size
    hex_height = np.sqrt(3) / 2.0 * crystal_size

    n_cols = int(width / (0.75 * hex_width))
    n_rows = int(height / hex_height)

    tile_numbers = []
    tile_x = []
    tile_y = []
    tile_z = []

    tile_id = 0

    for row in range(n_rows):

        for col in range(n_cols):

            x_local = (
                col * 0.75 * hex_width
                - width / 2.0
            )

            y_local = (
                row * hex_height
                - height / 2.0
            )

            # staggered rows
            if row % 2 == 1:
                x_local += 0.375 * hex_width

            # local coordinates relative to analyzer center
            relative_position = (
                x_local * horizontal_axis
                + y_local * vertical_axis
            )

            tile_numbers.append(tile_id)
            tile_x.append(float(relative_position[0]))
            tile_y.append(float(relative_position[1]))
            tile_z.append(float(relative_position[2]))

            tile_id += 1

    return {
        "center_vector": center.tolist(),
        "normal": normal.tolist(),
        "yaw_deg": yaw_deg,
        "pitch_deg": pitch_deg,
        "tile_number": tile_numbers,
        "tile_x": tile_x,
        "tile_y": tile_y,
        "tile_z": tile_z,
    }

In [3]:
result = analyzer_geometry_with_tiles(
    d_ll=1.98,
    d_lr=2.01,
    d_ur=2.03,
    horizontal_offset_deg=35.0,
    vertical_offset_deg=12.0,
    crystal_size=0.10
)

print(result.keys())
print(result['tile_x'])

dict_keys(['center_vector', 'normal', 'yaw_deg', 'pitch_deg', 'tile_number', 'tile_x', 'tile_y', 'tile_z'])
[0.07981350706578208, 0.0367952743394536, 0.043555000630972174, 0.0005367679046437042, 0.05031472692249074, 0.007296494196162273, 0.014056220487680836, -0.02896201223864763]


In [4]:


import json
import numpy as np


def analyzer_transformations_json(
    center_vector,
    yaw_deg,
    pitch_deg,
    name_prefix="ana_top1"
):
    """
    Generate NXtransformations JSON block for analyzer positioning.
    """

    x, y, z = center_vector

    transformations = {
        "name": "transformations",
        "type": "group",
        "children": [
            {
                "module": "dataset",
                "config": {
                    "name": f"x_translation",
                    "values": float(x),
                    "type": "double"
                },
                "attributes": [
                    {
                        "name": "vector",
                        "dtype": "double",
                        "values": [1.0, 0.0, 0.0]
                    },
                    {
                        "name": "depends_on",
                        "dtype": "string",
                        "values": "."
                    },
                    {
                        "name": "transformation_type",
                        "dtype": "string",
                        "values": "translation"
                    },
                    {
                        "name": "units",
                        "dtype": "string",
                        "values": "m"
                    }
                ]
            },
            {
                "module": "dataset",
                "config": {
                    "name": f"y_translation",
                    "values": float(y),
                    "type": "double"
                },
                "attributes": [
                    {
                        "name": "vector",
                        "dtype": "double",
                        "values": [0.0, 1.0, 0.0]
                    },
                    {
                        "name": "depends_on",
                        "dtype": "string",
                        "values": f"{name_prefix}x_translation"
                    },
                    {
                        "name": "transformation_type",
                        "dtype": "string",
                        "values": "translation"
                    },
                    {
                        "name": "units",
                        "dtype": "string",
                        "values": "m"
                    }
                ]
            },
            {
                "module": "dataset",
                "config": {
                    "name": f"z_translation",
                    "values": float(z),
                    "type": "double"
                },
                "attributes": [
                    {
                        "name": "vector",
                        "dtype": "double",
                        "values": [0.0, 0.0, 1.0]
                    },
                    {
                        "name": "depends_on",
                        "dtype": "string",
                        "values": f"{name_prefix}y_translation"
                    },
                    {
                        "name": "transformation_type",
                        "dtype": "string",
                        "values": "translation"
                    },
                    {
                        "name": "units",
                        "dtype": "string",
                        "values": "m"
                    }
                ]
            },
            {
                "module": "dataset",
                "config": {
                    "name": f"yaw",
                    "values": float(yaw_deg),
                    "type": "double"
                },
                "attributes": [
                    {
                        "name": "vector",
                        "dtype": "double",
                        "values": [0.0, 0.0, 1.0]
                    },
                    {
                        "name": "depends_on",
                        "dtype": "string",
                        "values": f"{name_prefix}z_translation"
                    },
                    {
                        "name": "transformation_type",
                        "dtype": "string",
                        "values": "rotation"
                    },
                    {
                        "name": "units",
                        "dtype": "string",
                        "values": "deg"
                    }
                ]
            },
            {
                "module": "dataset",
                "config": {
                    "name": f"pitch",
                    "values": float(pitch_deg),
                    "type": "double"
                },
                "attributes": [
                    {
                        "name": "vector",
                        "dtype": "double",
                        "values": [0.0, 1.0, 0.0]
                    },
                    {
                        "name": "depends_on",
                        "dtype": "string",
                        "values": f"{name_prefix}yaw"
                    },
                    {
                        "name": "transformation_type",
                        "dtype": "string",
                        "values": "rotation"
                    },
                    {
                        "name": "units",
                        "dtype": "string",
                        "values": "deg"
                    }
                ]
            }
        ],
        "attributes": [
            {
                "name": "NX_class",
                "dtype": "string",
                "values": "NXtransformations"
            }
        ]
    }

    return transformations

In [9]:
result = analyzer_geometry_with_tiles(
    d_ll=1.98,
    d_lr=2.01,
    d_ur=2.03,
    horizontal_offset_deg=35.0,
    vertical_offset_deg=12.0,
    crystal_size=0.10
)
print(result.keys())
print(result['tile_number'])
print(result['tile_x'])
print(result['tile_y'])
print(result['tile_z'])



dict_keys(['center_vector', 'normal', 'yaw_deg', 'pitch_deg', 'tile_number', 'tile_x', 'tile_y', 'tile_z'])
[0, 1, 2, 3, 4, 5, 6, 7]
[0.07981350706578208, 0.0367952743394536, 0.043555000630972174, 0.0005367679046437042, 0.05031472692249074, 0.007296494196162273, 0.014056220487680836, -0.02896201223864763]
[-0.05061706532025851, 0.010819338001415888, -0.030226497771583056, 0.031209905550091347, -0.071272333544582, -0.009835930222907602, -0.05088176599590655, 0.010554637325767852]
[-0.1709975041198336, -0.1709975041198336, -0.08628743703120623, -0.08628743703120623, -0.0015773699425788343, -0.0015773699425788343, 0.08313269714604858, 0.08313269714604858]


In [6]:
json_block = analyzer_transformations_json(
    center_vector=result["center_vector"],
    yaw_deg=result["yaw_deg"],
    pitch_deg=result["pitch_deg"],
    name_prefix='/entry/instrument/ana_top1/transformations/'
)

print(json.dumps(json_block, indent=2))

{
  "name": "transformations",
  "type": "group",
  "children": [
    {
      "module": "dataset",
      "config": {
        "name": "x_translation",
        "values": 1.59278779445798,
        "type": "double"
      },
      "attributes": [
        {
          "name": "vector",
          "dtype": "double",
          "values": [
            1.0,
            0.0,
            0.0
          ]
        },
        {
          "name": "depends_on",
          "dtype": "string",
          "values": "."
        },
        {
          "name": "transformation_type",
          "dtype": "string",
          "values": "translation"
        },
        {
          "name": "units",
          "dtype": "string",
          "values": "m"
        }
      ]
    },
    {
      "module": "dataset",
      "config": {
        "name": "y_translation",
        "values": 1.1517077595172238,
        "type": "double"
      },
      "attributes": [
        {
          "name": "vector",
          "dtype": "double",
     